# GRU history length 16 vs history length 1

This notebook compares the latest saved POMDP GRU-DQN run with a newly trained GRU-DQN that sees only one observation token at a time. The history-1 run uses its own checkpoint folder, `checkpoints_pomdp_history1/`.

Both runs use the same GRU size, observation encoding, seed, environment, optimizer settings, and evaluation rollouts. The history-1 control trains for 250,000 episodes; the reference run trained for 800,000. The history window is 16 tokens versus 1. A token includes the current partial observation and the controller's previous action; both models use the same epsilon-target memory feature. The return comparison therefore reflects both history length and training budget.

The training cell runs the full matching episode budget from the selected history-16 checkpoint. The evaluation uses paired seeds for each starting square and prints tables rather than opening graph windows.

In [4]:
from pathlib import Path
import importlib.util  # ensure importlib.util is available before importing oa.py
import json
import numpy as np
import pandas as pd
import torch

from controlsyn import ControlSynthesis
from dqn import GRUDQN
from mdp import GridMDP
from oa import OmegaAutomaton
from train_pomdp import STRUCTURE, LABEL

ROOT = Path.cwd()
POMDP_DIR = ROOT / "checkpoints_pomdp"
HISTORY1_DIR = ROOT / "checkpoints_pomdp_history1"
HISTORY1_DIR.mkdir(exist_ok=True)

reference_candidates = sorted(
    POMDP_DIR.glob("dqn_*.pt"), key=lambda path: path.stat().st_mtime
)
assert reference_candidates, f"No POMDP checkpoints found in {POMDP_DIR}"
REFERENCE_CHECKPOINT = reference_candidates[-1]
reference_payload = torch.load(REFERENCE_CHECKPOINT, map_location="cpu", weights_only=True)
REFERENCE_META = reference_payload["metadata"]
HISTORY1_EPISODES = 250_000
assert REFERENCE_META["architecture"] == "gru"
assert REFERENCE_META["history_length"] == 16, REFERENCE_META["history_length"]

print("Reference checkpoint:", REFERENCE_CHECKPOINT.name)
print("Run ID:", REFERENCE_META["run_id"])
print("Reference episodes / history-1 episodes / horizon:", REFERENCE_META["episodes"], "/", HISTORY1_EPISODES, "/", REFERENCE_META["steps_per_episode"])
print("History length:", REFERENCE_META["history_length"])
print("Seed:", REFERENCE_META["seed"])

Reference checkpoint: dqn_20261017T121024.314516Z_K800000_T100_1df3146.pt
Run ID: 20261017T121024.314516Z_K800000_T100_1df3146
Reference episodes / history-1 episodes / horizon: 800000 / 250000 / 100
History length: 16
Seed: 17


In [5]:
grid_mdp = GridMDP(
    shape=tuple(REFERENCE_META["mdp_shape"]),
    structure=STRUCTURE,
    label=LABEL,
    observation_error=float(REFERENCE_META["observation_error"]),
)
oa = OmegaAutomaton("(F G a | F G b) & G !c", "ldba")
csrl = ControlSynthesis(
    grid_mdp,
    oa,
    discount_nonaccepting=float(REFERENCE_META["discount_nonaccepting"]),
    discount_accepting=float(REFERENCE_META["discount_accepting"]),
)

valid_starts = [
    (int(r), int(c)) for r, c in grid_mdp.states()
    if grid_mdp.structure[r, c] != "B"
]
global_epsilon_actions = sorted({
    len(grid_mdp.A) + destination
    for outgoing in oa.eps
    for destination in outgoing
})
global_action_ids = list(range(len(grid_mdp.A))) + global_epsilon_actions
print(f"{len(valid_starts)} traversable start squares; legal global actions: {global_action_ids}")

19 traversable start squares; legal global actions: [0, 1, 2, 3, 5, 6]


## Train the history-1 control

The checkpoint directory is separate so the short-history weights and metadata cannot overwrite the history-16 run. Training settings are copied from that run's metadata, with `history_length=1` and a 250,000-episode training budget.

In [6]:
TRAIN_HISTORY1 = True

if TRAIN_HISTORY1:
    _, one_step_policy = csrl.deep_q_learning(
        T=int(REFERENCE_META["steps_per_episode"]),
        K=HISTORY1_EPISODES,
        seed=int(REFERENCE_META["seed"]),
        learning_rate=float(REFERENCE_META["learning_rate"]),
        epsilon_floor=float(REFERENCE_META["epsilon_floor"]),
        epsilon_decay_fraction=float(REFERENCE_META["epsilon_decay_fraction"]),
        automaton_action_probability=float(REFERENCE_META["automaton_action_probability"]),
        architecture="gru",
        history_length=1,
        gru_hidden_size=int(REFERENCE_META["gru_hidden_size"]),
        partial_observation=True,
        batch_size=int(REFERENCE_META["batch_size"]),
        replay_capacity=int(REFERENCE_META["replay_capacity"]),
        target_update_every=int(REFERENCE_META["target_update_every"]),
        checkpoint_dir=str(HISTORY1_DIR),
        return_model=True,
    )
    HISTORY1_CHECKPOINT = Path(csrl.last_checkpoint_path)

    manifest = {
        "comparison": "GRU history length 16 versus history length 1",
        "reference_checkpoint": str(REFERENCE_CHECKPOINT),
        "reference_run_id": REFERENCE_META["run_id"],
        "history1_checkpoint": str(HISTORY1_CHECKPOINT),
        "history1_run_id": HISTORY1_CHECKPOINT.stem.removeprefix("dqn_"),
        "history1_episodes": HISTORY1_EPISODES,
        "changed_parameters": ["history_length", "episodes"],
        "history_length_reference": 16,
        "history_length_control": 1,
    }
    (HISTORY1_DIR / "comparison_manifest.json").write_text(
        json.dumps(manifest, indent=2) + "\n"
    )
    print("History-1 checkpoint:", HISTORY1_CHECKPOINT)
else:
    candidates = sorted(HISTORY1_DIR.glob("dqn_*.pt"), key=lambda path: path.stat().st_mtime)
    assert candidates, f"No history-1 checkpoint found in {HISTORY1_DIR}"
    HISTORY1_CHECKPOINT = candidates[-1]
    print("Using existing history-1 checkpoint:", HISTORY1_CHECKPOINT)

Episode 0/250000
Episode 2500/250000; trailing 1000-episode return=0.073258
Episode 5000/250000; trailing 1000-episode return=0.074417
Episode 7500/250000; trailing 1000-episode return=0.078338
Episode 10000/250000; trailing 1000-episode return=0.070723
Episode 12500/250000; trailing 1000-episode return=0.087503
Episode 15000/250000; trailing 1000-episode return=0.091582
Episode 17500/250000; trailing 1000-episode return=0.086586
Episode 20000/250000; trailing 1000-episode return=0.095506
Episode 22500/250000; trailing 1000-episode return=0.106599
Episode 25000/250000; trailing 1000-episode return=0.108987
Episode 27500/250000; trailing 1000-episode return=0.104710
Episode 30000/250000; trailing 1000-episode return=0.121718
Episode 32500/250000; trailing 1000-episode return=0.129120
Episode 35000/250000; trailing 1000-episode return=0.129103
Episode 37500/250000; trailing 1000-episode return=0.143032
Episode 40000/250000; trailing 1000-episode return=0.123636
Episode 42500/250000; trai

## Load both policies

Both policies use the same GRU architecture and feature width. During evaluation each receives its own trained history window (16 or 1), with the same greedy action selection and action-availability rule.

In [7]:
history1_payload = torch.load(HISTORY1_CHECKPOINT, map_location="cpu", weights_only=True)
HISTORY1_META = history1_payload["metadata"]
assert HISTORY1_META["architecture"] == "gru"
assert HISTORY1_META["history_length"] == 1
assert HISTORY1_META["episodes"] == HISTORY1_EPISODES
assert HISTORY1_META["steps_per_episode"] == REFERENCE_META["steps_per_episode"]
assert HISTORY1_META["seed"] == REFERENCE_META["seed"]
assert HISTORY1_META["learning_rate"] == REFERENCE_META["learning_rate"]
assert HISTORY1_META["batch_size"] == REFERENCE_META["batch_size"]
assert HISTORY1_META["target_update_every"] == REFERENCE_META["target_update_every"]


def load_gru(payload):
    meta = payload["metadata"]
    model = GRUDQN(
        int(meta["n_observations"]),
        int(meta["n_actions"]),
        hidden_size=int(meta["gru_hidden_size"]),
        num_layers=int(meta["gru_num_layers"]),
    )
    model.load_state_dict(payload["policy_state_dict"])
    model.eval()
    return model


models = {
    "history_16": load_gru(reference_payload),
    "history_1": load_gru(history1_payload),
}
metadata_by_model = {
    "history_16": REFERENCE_META,
    "history_1": HISTORY1_META,
}
print("Loaded history-16 and history-1 policies.")

Loaded history-16 and history-1 policies.


## Matched greedy evaluation

For each starting square and rollout number, both policies get the same NumPy seed. This pairs the sensor-noise and movement randomness as closely as possible, while allowing the policies to take different paths. Returns use the environment's reward-conditioned discounts and the same 1,000-step evaluation horizon used by the existing POMDP evaluator.

In [8]:
EVAL_RUNS_PER_START = 100
EVAL_HORIZON = 1_000
EVAL_SEED = 17


def rejecting_sink_states():
    return {
        q for q in range(oa.shape[1])
        if not oa.eps[q]
        and oa.delta[q]
        and all(destination == q for destination in oa.delta[q].values())
        and not any(
            status is True
            for statuses in oa.acc[q].values()
            for status in statuses
        )
    }


REJECTING_SINKS = rejecting_sink_states()


def greedy_episode(model, meta, start_square, episode_seed, horizon=EVAL_HORIZON):
    np.random.seed(int(episode_seed))
    state = (0, oa.q0, int(start_square[0]), int(start_square[1]))
    observations = [csrl.observe_observation(state)]
    previous_actions = [None]
    epsilon_target = None
    total_return = 0.0
    discount = 1.0
    first_action = None

    for step in range(horizon):
        history_length = int(meta["history_length"])
        encoded_history = csrl.encode_observation_history(
            observations,
            history_length=history_length,
            previous_actions=previous_actions,
            partial_observation=True,
            epsilon_committed=epsilon_target is not None,
            epsilon_target=epsilon_target,
            epsilon_commitment_mode=meta.get(
                "recurrent_epsilon_commitment_mode", "target_one_hot"
            ),
        ).unsqueeze(0)
        with torch.no_grad():
            q_values = model(encoded_history).squeeze(0).cpu().numpy()

        if bool(meta.get("epsilon_commit_mask_from_history", False)) and epsilon_target is not None:
            allowed = list(range(len(grid_mdp.A)))
        else:
            allowed = global_action_ids
        q_values[[action for action in range(len(q_values)) if action not in allowed]] = -np.inf
        action = int(np.argmax(q_values))
        if first_action is None:
            first_action = action

        reward = float(csrl.reward[state])
        total_return += discount * reward
        discount *= (
            csrl.discount_accepting if reward > 0
            else csrl.discount_nonaccepting
        )

        if action in csrl.A[state]:
            successors, probabilities = csrl.transition_probs[state][action]
        elif action in global_epsilon_actions:
            # A globally available epsilon attempt with no edge here is a no-op.
            successors, probabilities = [state], [1.0]
        else:
            raise RuntimeError(f"Unexpected unavailable action {action} at {state}")
        state = successors[np.random.choice(len(successors), p=probabilities)]

        if action in global_epsilon_actions and epsilon_target is None:
            epsilon_target = action - len(grid_mdp.A)
        observations.append(csrl.observe_observation(state))
        previous_actions.append(action)
        observations = observations[-history_length:]
        previous_actions = previous_actions[-history_length:]
        if state[1] in REJECTING_SINKS:
            break

    return total_return, first_action


paired_records = []
for start_index, square in enumerate(valid_starts):
    for rollout in range(EVAL_RUNS_PER_START):
        pair_seed = EVAL_SEED + start_index * 1_000_003 + rollout
        result = {}
        for name in ("history_16", "history_1"):
            result[name] = greedy_episode(
                models[name], metadata_by_model[name], square, pair_seed
            )
        paired_records.append({
            "square": square,
            "label": tuple(grid_mdp.label[square]),
            "rollout": rollout,
            "return_16": result["history_16"][0],
            "return_1": result["history_1"][0],
            "first_action_16": result["history_16"][1],
            "first_action_1": result["history_1"][1],
        })

paired = pd.DataFrame(paired_records)
print(f"Evaluated {len(paired)} paired rollouts ({EVAL_RUNS_PER_START} per start square).")

Evaluated 1900 paired rollouts (100 per start square).


In [9]:
start_rows = []
for square, group in paired.groupby("square", sort=True):
    differences = group["return_16"] - group["return_1"]
    delta = float(differences.mean())
    se = float(differences.std(ddof=1) / np.sqrt(len(differences))) if len(differences) > 1 else 0.0
    start_rows.append({
        "square": square,
        "row": square[0],
        "col": square[1],
        "label": "".join(grid_mdp.label[square]) or "empty",
        "mean_return_h16": float(group["return_16"].mean()),
        "mean_return_h1": float(group["return_1"].mean()),
        "paired_delta_h16_minus_h1": delta,
        "delta_ci95_low": delta - 1.96 * se,
        "delta_ci95_high": delta + 1.96 * se,
    })

start_results = pd.DataFrame(start_rows)
display(start_results.round(4))

safe = paired[paired["label"] != ("c",)]
for name, subset in (("all starts", paired), ("non-c starts", safe)):
    delta = subset["return_16"] - subset["return_1"]
    se = float(delta.std(ddof=1) / np.sqrt(len(delta)))
    print(
        f"{name}: history-16 mean={subset['return_16'].mean():.5f}; "
        f"history-1 mean={subset['return_1'].mean():.5f}; "
        f"paired delta={delta.mean():+.5f} "
        f"(95% CI {delta.mean()-1.96*se:+.5f} to {delta.mean()+1.96*se:+.5f})"
    )

print("Positive paired delta favors history length 16.")

,square,row,col,label,mean_return_h16,mean_return_h1,paired_delta_h16_minus_h1,delta_ci95_low,delta_ci95_high
0,"(0, 0)",0,0,empty,0.1386,0.2524,-0.1138,-0.2193,-0.0083
1,"(0, 1)",0,1,empty,0.1238,0.2449,-0.1211,-0.2228,-0.0194
2,"(0, 2)",0,2,c,0.0000,0.0000,0.0000,0.0000,0.0000
3,"(0, 3)",0,3,empty,0.9195,0.7299,0.1896,0.0828,0.2964
4,"(1, 0)",1,0,empty,0.0673,0.2549,-0.1877,-0.2814,-0.0940
5,"(1, 1)",1,1,empty,0.0580,0.2691,-0.2112,-0.3077,-0.1147
6,"(1, 2)",1,2,a,0.6498,0.7099,-0.0601,-0.1068,-0.0133
7,"(1, 3)",1,3,b,0.9995,0.9998,-0.0003,-0.0004,-0.0003
8,"(2, 1)",2,1,empty,0.1771,0.3796,-0.2024,-0.3127,-0.0922
9,"(2, 2)",2,2,c,0.0000,0.0000,0.0000,0.0000,0.0000


all starts: history-16 mean=0.48852; history-1 mean=0.52000; paired delta=-0.03147 (95% CI -0.04831 to -0.01464)
non-c starts: history-16 mean=0.61880; history-1 mean=0.65866; paired delta=-0.03987 (95% CI -0.06117 to -0.01856)
Positive paired delta favors history length 16.


In [10]:
# Compact grid view of the per-square mean-return difference (history 16 minus history 1).
delta_grid = start_results.pivot(
    index="row", columns="col", values="paired_delta_h16_minus_h1"
)
display(delta_grid.round(3))

col,0,1,2,3
row,,,,
0,-0.114,-0.121,0.00,0.19
1,-0.188,-0.211,-0.06,-0.00
2,NaN,-0.202,0.00,-0.03
3,-0.000,-0.000,-0.00,0.06
4,0.040,0.000,0.04,0.00
